# Exercise 4

In [ ]:
from functions import RandomnessTests
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import random
random.seed(42)
import time
import tracemalloc
np.random.seed(42)
rng = np.random.default_rng(42)
from scipy.special import factorial
from scipy import stats
import heapq

## Part 1

In [ ]:
def erlang_b(A, m):
    num = (A**m) / factorial(m)
    den = sum([(A**i) / factorial(i) for i in range(m + 1)])
    return num / den

def mean_confidence_interval(data, confidence=0.95):
    a = 1.0 * np.array(data)
    n = len(a)
    m, se = np.mean(a), stats.sem(a)
    h = se * stats.t.ppf((1 + confidence) / 2., n-1)
    return m, m-h, m+h


In [ ]:
def sim_block_system(m, num_customers, arrival_gen, service_gen):
    clock = 0.0
    blocked_count = 0
    # priority kø
    servers = [] 
    
    for _ in range(num_customers):
        clock += arrival_gen()
        
        while servers and servers[0] <= clock:
            heapq.heappop(servers)
            
        if len(servers) < m:
            service_time = service_gen()
            heapq.heappush(servers, clock + service_time)
        else:
            blocked_count += 1
            
    return blocked_count / num_customers

In [ ]:
np.random.seed(42)
m = 10
mean_service = 8.0
mean_interarrival = 1.0
num_customers = 10000
num_runs = 10

arrival_gen = lambda: np.random.exponential(mean_interarrival)
service_gen = lambda: np.random.exponential(mean_service)

results_p1 = [sim_block_system(m, num_customers, arrival_gen, service_gen) for _ in range(num_runs)]
mean_p1, ci_low_p1, ci_high_p1 = mean_confidence_interval(results_p1)
exact_p1 = erlang_b(8.0, 10)

print(f"  Observed Blocked Fraction: {mean_p1:.4f} (95% CI: [{ci_low_p1:.4f}, {ci_high_p1:.4f}])")
print(f"  Exact Erlang B Solution:   {exact_p1:.4f}")

## Part 2

### a - Erlang arrivals

In [ ]:
np.random.seed(42)  

k_erlang = 2
erlang_arrival = lambda: np.random.gamma(k_erlang, 1.0/k_erlang)

results_p2a = [sim_block_system(m, num_customers, erlang_arrival, service_gen) for _ in range(num_runs)]
mean_p2a, ci_low_p2a, ci_high_p2a = mean_confidence_interval(results_p2a)

print(f"  Observed Blocked Fraction: {mean_p2a:.4f} (95% CI: [{ci_low_p2a:.4f}, {ci_high_p2a:.4f}])")

### b - Hyperexponential arrivals

In [ ]:
def hyperexponential_arrival():
    if np.random.rand() < 0.8:
        return np.random.exponential(1/0.8333)
    else:
        return np.random.exponential(1/5.0)

In [ ]:
results_p2b = [sim_block_system(m, num_customers, hyperexponential_arrival, service_gen) for _ in range(num_runs)]
mean_p2b, ci_low_p2b, ci_high_p2b = mean_confidence_interval(results_p2b)

print(f"  Observed Blocked Fraction: {mean_p2b:.4f} (95% CI: [{ci_low_p2b:.4f}, {ci_high_p2b:.4f}])")

## Part 3

### a - constant service time

In [ ]:
np.random.seed(42)
constant_service = lambda: 8.0

results_p3a = [sim_block_system(m, num_customers, arrival_gen, constant_service) for _ in range(num_runs)]
mean_p3a, ci_low_p3a, ci_high_p3a = mean_confidence_interval(results_p3a)

print(f"  Observed Blocked Fraction: {mean_p3a:.4f} (95% CI: [{ci_low_p3a:.4f}, {ci_high_p3a:.4f}])")

### b - Pareto distributed service time

In [ ]:
np.random.seed(42)

def pareto_service(k, mean_target=8.0):
    beta = mean_target * (k - 1) / k
    return (np.random.pareto(k) + 1) * beta

pareto_gen = lambda: pareto_service(2.05)
results_p3b = [sim_block_system(m, num_customers, arrival_gen, pareto_gen) for _ in range(num_runs)]
mean_p3b, ci_low_p3b, ci_high_p3b = mean_confidence_interval(results_p3b)

print(f"  Observed Blocked Fraction: {mean_p3b:.4f} (95% CI: [{ci_low_p3b:.4f}, {ci_high_p3b:.4f}])")

### c - choose one or two other distributions

In [ ]:
np.random.seed(42)

def gamma_service(shape=3.0, mean_target=8.0):
    scale = mean_target / shape
    return np.random.gamma(shape, scale)


def lognormal_service(sigma=0.6, mean_target=8.0):
    mu = np.log(mean_target) - 0.5 * sigma**2
    return np.random.lognormal(mean=mu, sigma=sigma)

service_gamma = lambda: gamma_service(shape=3.0, mean_target=8.0)
service_lognormal = lambda: lognormal_service(sigma=0.6, mean_target=8.0)

results_p3c_gamma = [sim_block_system(m, num_customers, arrival_gen, service_gamma) for _ in range(num_runs)]
mean_p3c_gamma, ci_low_p3c_gamma, ci_high_p3c_gamma = mean_confidence_interval(results_p3c_gamma)

print(f"Gamma service:     Observed Blocked Fraction: {mean_p3c_gamma:.4f} (95% CI: [{ci_low_p3c_gamma:.4f}, {ci_high_p3c_gamma:.4f}])")

results_p3c_lognormal = [sim_block_system(m, num_customers, arrival_gen, service_lognormal) for _ in range(num_runs)]
mean_p3c_lognormal, ci_low_p3c_lognormal, ci_high_p3c_lognormal = mean_confidence_interval(results_p3c_lognormal)

print(f"Lognormal service: Observed Blocked Fraction: {mean_p3c_lognormal:.4f} (95% CI: [{ci_low_p3c_lognormal:.4f}, {ci_high_p3c_lognormal:.4f}])")

## Part 5

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

labels = [
    "Part 1\nExp/Exp",
    "Part 2a\nErlang arrivals",
    "Part 2b\nHyperexp arrivals",
    "Part 3a\nConstant service",
    "Part 3b\nPareto service",
    "Part 3c\nGamma service",
    "Part 3c\nLognormal service",
]

means = [
    mean_p1,
    mean_p2a,
    mean_p2b,
    mean_p3a,
    mean_p3b,
    mean_p3c_gamma,
    mean_p3c_lognormal,
]
lows = [
    mean_p1 - ci_low_p1,
    mean_p2a - ci_low_p2a,
    mean_p2b - ci_low_p2b,
    mean_p3a - ci_low_p3a,
    mean_p3b - ci_low_p3b,
    mean_p3c_gamma - ci_low_p3c_gamma,
    mean_p3c_lognormal - ci_low_p3c_lognormal,
]
highs = [
    ci_high_p1 - mean_p1,
    ci_high_p2a - mean_p2a,
    ci_high_p2b - mean_p2b,
    ci_high_p3a - mean_p3a,
    ci_high_p3b - mean_p3b,
    ci_high_p3c_gamma - mean_p3c_gamma,
    ci_high_p3c_lognormal - mean_p3c_lognormal,
]

fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(labels))
ax.bar(x, means, yerr=[lows, highs], capsize=5, color=["#4C78A8"] * len(labels), alpha=0.85)
ax.axhline(exact_p1, color="#FF5733", linestyle="--", label="Exact Erlang B")
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=20, ha="right")
ax.set_ylabel("Blocked fraction")
ax.set_title("Comparison of distributions with 95% confidence intervals")
ax.set_ylim(0, max(max(highs) + max(means), exact_p1) * 1.25)
ci_handle = Line2D([0], [0], color="#4C78A8", linewidth=2, marker="|", markersize=12, label="95% CI")
exact_handle = Line2D([0], [0], color="#FF5733", linestyle="--", linewidth=2, label="Exact Erlang B")
ax.legend(handles=[ci_handle, exact_handle])
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()